# Validacion Multi-Region HAB (5 ecosistemas)

**Regiones:** Okeechobee, Tampa Bay, Cajon, Golfo de Fonseca, Lago de Yojoa

- **In-situ:** Okeechobee, Tampa Bay, Yojoa (clorofila + etiqueta HAB >= 10 ug/L)
- **Espectral (GeoTIFF):** las 5 regiones (pixeles de agua, correlacion prob-NDCI)
- **Adaptacion de dominio:** z-score espectral Florida <- Yojoa en Honduras


In [ ]:
import os, json, pandas as pd, matplotlib.pyplot as plt

BASE = r'C:\\Users\\JC\\Desktop\\Tesis'
OUT = os.path.join(BASE, 'validacion_mapas')

# Cargar resultados de validacion precalculados (generados por validar_chl.py)
# Estos archivos tienen las correlaciones entre probabilidad HAB y clorofila
res = json.load(open(os.path.join(OUT, 'resumen_validacion_regiones.json'), encoding='utf-8'))
df = pd.read_csv(os.path.join(OUT, 'validacion_por_region.csv'))
print(df[['region','validation_type','n_muestras','nn_auc','xgb_auc','nn_pearson_chl','corr_prob_ndci']].to_string(index=False))


In [ ]:
# Validacion con datos in-situ (donde hay clorofila medida en campo)
insitu = df[df['validation_type']=='in_situ'].dropna(subset=['nn_auc'])
if len(insitu):
    fig, ax = plt.subplots(figsize=(10,5))
    x = range(len(insitu)); w=0.35
    # Grafico de barras comparando AUC de los dos modelos por region
    ax.bar([i-w/2 for i in x], insitu['nn_auc'], w, label='HABNet1Dv2', color='steelblue')
    ax.bar([i+w/2 for i in x], insitu['xgb_auc'], w, label='XGBoost', color='darkorange')
    ax.set_xticks(list(x)); ax.set_xticklabels(insitu['region'], rotation=15)
    ax.set_ylim(0,1.05); ax.set_ylabel('AUC-ROC'); ax.set_title('Validacion in-situ por region')
    ax.legend(); ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, 'validacion_auc_por_region.png'), dpi=150)
    plt.show()

# Validacion espectral: correlacion entre probabilidad HAB y NDCI en los GeoTIFFs
# NDCI es el indice de clorofila mas usado, si nuestra probabilidad se correlaciona con el, es buena senial
geo = df[df['validation_type']=='espectral_geotiff']
print('\nValidacion espectral (GeoTIFF):')
print(geo[['region','n_pixeles','corr_prob_ndci','prob_mean','hab_pct_umbral_0.5']].to_string(index=False))
